# 3. Dimensionality reduction: KinCore labels and ligands <a id="3"></a>
In this section we will apply dimensionality reduction to our coarse-grained representation of selected activation loops.


## Table of contents

- [3.2 Conformational classification](#23)
  - [KinCore ligand types by activation state](#ligand-types-activation)
- [3.3 Ligand alignment](#24lig)


## Backend map

How this notebook connects to `workflow/` modules:

```mermaid
flowchart LR
  nb["06-KinCoreLabelsAndLigands"]
  m0["workflow.DunbrackAssignment"]
  nb --> m0
  m1["workflow.align"]
  nb --> m1
  m2["workflow.utilities"]
  nb --> m2
```


![State of the workflow](images/DimensionalityReduction.png)

To get started, let's load some packages!

In [ ]:
from IPython.display import display, HTML

from workflow.DunbrackAssignment import DunbrackWorkflow
from workflow.align import Alignment
from workflow.utilities import PDBDownloader
from workflow.utilities import (
    count_pdb_files,
    braf_res,
    clear_and_make,
    make_seg,
    copy_filtered_pdbs,
    copy_cg_chain_small_molecules,
)


## 3.2 Conformational classification  <a id="23"></a>
Here we investigate the conformational diversity of our kinase domains by applying the classification developed by the Dunbrack's group.


The class `DunbrackWorkflow` enables conformational classification with `KinCore` on the protein–small-molecule complexes in `Results/CG_chain_small_molecules/` (fitted-set chains mirrored from `motif_filtered_small_molecules/`).

**When KinCore fails**, structures are marked with `'failed'` status. This happens when:
- KinCore cannot find the DFG or C-helix motifs
- Structure has missing residues in critical regions
- Non-standard kinase fold
- Structure quality issues


In [ ]:
from workflow.DunbrackAssignment import DunbrackWorkflow

paths = DunbrackWorkflow.build_output_paths(output_dir="Results/dunbrack_assignments")
DUNBRACK_OUTPUT_DIR = paths["output_dir"]
KINCORE_ASSIGNMENTS_CSV = paths["assignments_csv"]
TRUE_LIGAND_CSV = paths["true_ligand_csv"]
CONFORMATION_PLOT_PNG = paths["conformation_plot_png"]
ACTIVATION_LIGAND_TYPE_HIST_PNG = paths["activation_ligand_type_histogram_png"]
LIGAND_TYPES_BY_ACTIVATION_PNG = paths["ligand_types_by_activation_png"]
LIGAND_TYPES_BY_ACTIVATION_CSV = paths["ligand_types_by_activation_csv"]

FORCE_KINCORE = False


In [ ]:

_ = DunbrackWorkflow.ensure_assignments_cached(
    input_dir="Results/CG_chain_small_molecules/",
    output_dir=DUNBRACK_OUTPUT_DIR,
    kincore_dir="/home/marmatt/Documents/Kincore-standalone",
    assignments_filename="kinase_conformation_assignments.csv",
    force=FORCE_KINCORE,
)


Let's now print some information about the KinCore analysis.


In [ ]:
from workflow.DunbrackAssignment import DunbrackWorkflow

ligand_report_df = DunbrackWorkflow.report_from_assignments_csv(
    assignments_csv=KINCORE_ASSIGNMENTS_CSV,
    true_ligand_csv=TRUE_LIGAND_CSV,
)


We copy protein–small-molecule complexes that pass KinCore **true-ligand** filtering (`true_has_ligand`) from `Results/CG_chain_small_molecules/` into `Results/CG_chain_ligand/` for the ligand-focused alignment section below.


In [ ]:
from workflow.DunbrackAssignment import DunbrackWorkflow

LIGANDS_SRC = "Results/CG_chain_small_molecules/"
LIGANDS_DST = "Results/CG_chain_ligand/"

_ = DunbrackWorkflow.copy_true_ligand_structures(
    pdb_dir=LIGANDS_SRC,
    true_ligand_csv=TRUE_LIGAND_CSV,
    out_dir=LIGANDS_DST,
)


We can investigate if KinCore predictions of activation state correlate with the presence of ligands in the curated dataset.


In [ ]:
_ = DunbrackWorkflow.plot_activation_vs_ligand_type_histogram(
    assignments_csv=KINCORE_ASSIGNMENTS_CSV,
    true_ligand_csv=TRUE_LIGAND_CSV,
    output_png=ACTIVATION_LIGAND_TYPE_HIST_PNG,
)


### KinCore ligand types by activation state <a id="ligand-types-activation"></a>

For each KinCore ligand class (**Apo**, **Type1**, **Type1.5**, **Type2**, **Type3**, **Allosteric**), we plot the fraction of **Active** vs **Inactive** structures that carry that type.

- **Y-axis:** percentage **within each activation class** (not global), i.e. `#(class with type T) / #(all structures in class) × 100`.
- **Activation labels** come from KinCore `conformation_description` (Active / Inactive); **Unknown** conformations are excluded.
- **Multi-label** structures (e.g. `Allosteric,Type1`) count toward every listed type.

The Type1/Type2 histogram above is kept separately for the true-ligand Type1 vs Type2 counts.


In [ ]:
from workflow.DunbrackAssignment import DunbrackWorkflow

summary_df = DunbrackWorkflow.plot_ligand_types_by_activation(
    assignments_csv=KINCORE_ASSIGNMENTS_CSV,
    output_png=LIGAND_TYPES_BY_ACTIVATION_PNG,
    output_csv=LIGAND_TYPES_BY_ACTIVATION_CSV,
    show=True,
)
display(summary_df)


In [ ]:
from workflow.DunbrackAssignment import DunbrackWorkflow

type1_sel_df = DunbrackWorkflow.summarize_type1_chains(
    pdb_dir="Results/CG_chain_small_molecules/",
    true_ligand_csv=TRUE_LIGAND_CSV,
)

display(type1_sel_df.head(50))
print("(showing first 50 rows)")


We decide to narrow down the search to ATP and ATP analogue ligands present in our dataset and check how they relate to activation state prediction by Kincore.


In [ ]:
from workflow.DunbrackAssignment import DunbrackWorkflow

atp_manifest_df = DunbrackWorkflow.copy_atp_analogue_structures(
    pdb_dir="Results/CG_chain_small_molecules/",
    true_ligand_csv=TRUE_LIGAND_CSV,
    combined_out_dir="Results/CG_chain_S_MOLECULE_ATP+analogues",
)

display(atp_manifest_df.head(50))
print("(showing first 50 rows)")


Let's now visualise all of the metadata extracted from `KinCore`.


In [ ]:
from workflow.DunbrackAssignment import DunbrackWorkflow

_ = DunbrackWorkflow.show_or_plot_conformation_distribution(
    assignments_csv=KINCORE_ASSIGNMENTS_CSV,
    output_png=CONFORMATION_PLOT_PNG,
    show=True,
    print_dunbrack_summary=True,
)


## 3.3 Ligand alignment  <a id="24lig"></a>
In this section we apply structural superposition to the structures in `Results/CG_chain_ligand/` (the KinCore true-ligand subset).


We align the true-ligand protein–ligand complexes from `Results/CG_chain_ligand/` to the same reference used for the protein-only chains (`6UAN_chainD`), using DFG+APE motif Cα atoms via `Alignment.process_pymol_alignment`. Aligned complexes are written to `Results/CG_chain_ligand_aligned/`.


In [ ]:
from workflow.align import Alignment

aligner = Alignment()
reference_pdb = "6UAN_chainD.pdb"
LIGANDS_ALIGNED_DIR = "Results/CG_chain_ligand_aligned/"

aligner.process_pymol_alignment(
    pdb_dir="Results/CG_chain_ligand/",
    reference_pdb=reference_pdb,
    output_dir=LIGANDS_ALIGNED_DIR,
    ref_name="6UAN_chainD",
    quiet=True,
)


In [ ]:
pdb_directory = "Results/CG_chain_ligand/"
pdb_directory2 = "Results/CG_chain_ligand_aligned/"
pdb_count = count_pdb_files(pdb_directory)
pdb_count2 = count_pdb_files(pdb_directory2)

print(f"There are {pdb_count} PDB files in the directory '{pdb_directory}'.")
print(f"There are {pdb_count2} PDB files in the directory '{pdb_directory2}'.")
if pdb_count2 == 0:
    raise RuntimeError(
        "No aligned CG chain–ligand complexes were written. "
        "Confirm MDAnalysis is installed in this kernel, then re-run the alignment cell above."
    )
